In [36]:
!pip install wikiextractor -q


In [37]:
from pathlib import Path

p = Path("/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py")
backup = p.with_suffix(".py.bak")

# 백업
if not backup.exists():
    backup.write_text(p.read_text(encoding="utf-8"), encoding="utf-8")

lines = p.read_text(encoding="utf-8").splitlines(keepends=True)

new_lines = []
i = 0
patched_1 = False
patched_2 = False

while i < len(lines):
    s = lines[i].lstrip()

    if s.startswith("ExtLinkBracketedRegex = re.compile("):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + "ExtLinkBracketedRegex = re.compile(\n",
            indent + "    '\\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\\s*([^\\]\\x00-\\x08\\x0a-\\x1F]*?)\\]',\n",
            indent + "    re.S | re.U | re.I)\n",
        ])
        i += 3
        patched_1 = True
        continue

    if s.startswith("EXT_IMAGE_REGEX = re.compile("):
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        new_lines.extend([
            indent + "EXT_IMAGE_REGEX = re.compile(\n",
            indent + '    r"""^(http://|https://)([^][<>"\\x00-\\x20\\x7F\\s]+)\n',
            indent + '    /([A-Za-z0-9_.,~%\\-+&;#*?!=()@\\x80-\\xFF]+)\\.(gif|png|jpg|jpeg)$""",\n',
            indent + "    re.X | re.S | re.U | re.I)\n",
        ])
        i += 4
        patched_2 = True
        continue

    new_lines.append(lines[i])
    i += 1

p.write_text("".join(new_lines), encoding="utf-8")

print("backup:", backup)
print("patched ExtLinkBracketedRegex:", patched_1)
print("patched EXT_IMAGE_REGEX:", patched_2)
print("done")

backup: /usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py.bak
patched ExtLinkBracketedRegex: True
patched EXT_IMAGE_REGEX: True
done


In [38]:
!python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text

/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:33: SyntaxWarning: invalid escape sequence '\w'
  tailRE = re.compile('\w+')
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:171: SyntaxWarning: invalid escape sequence '\.'
  text = re.sub(u' (,:\.\)\]»)', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:172: SyntaxWarning: invalid escape sequence '\['
  text = re.sub(u'(\[\(«) ', r'\1', text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:379: SyntaxWarning: invalid escape sequence '\['
  '\[((' + '|'.join(wgUrlProtocols) + ')' + EXT_LINK_URL_CLASS + r'+)\s*([^\]\x00-\x08\x0a-\x1F]*?)\]',
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:733: SyntaxWarning: invalid escape sequence '\w'
  return re.sub("&#?(\w+);", fixup, text)
/usr/local/lib/python3.12/dist-packages/wikiextractor/extract.py:1278: SyntaxWarning: invalid escape sequence '\['
  reOpen = re.compile('{{2,}|\[{2,}')
/usr/local/lib/

In [39]:
# 코랩환경 기준

# 데이터 파싱 을 위한 패키지 설치
!pip install wikiextractor

In [40]:
# 위키피디아 데이터 다운로드, 전처리에서 사용할 형태소 분석기 (Mecab)설치

# !git clone https://github.com//SOMJANG/Mecab-ko-for-Google-colab.github
# %cd Mecab-ko-for-Google-colab.github
# !bash install_mecab-ko_on_colab190912.sh

!git clone https://github.com/SOMJANG/Mecab-ko-for-Google-Colab
%cd Mecab-ko-for-Google-Colab
!bash install_mecab-ko_on_colab190912.sh




fatal: destination path 'Mecab-ko-for-Google-Colab' already exists and is not an empty directory.
/content/Mecab-ko-for-Google-Colab
Installing konlpy.....
Done
Installing mecab-0.996-ko-0.9.2.tar.gz.....
from https://bitbucket.org/eunjeon/mecab-ko/downloads/mecab-0.996-ko-0.9.2.tar.gz
--2026-03-12 14:22:27--  https://bitbucket.org/eunjeon/mecab-ko/downloads/mecab-0.996-ko-0.9.2.tar.gz
Resolving bitbucket.org (bitbucket.org)... 104.192.142.24, 104.192.142.26, 104.192.142.25, ...
Connecting to bitbucket.org (bitbucket.org)|104.192.142.24|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-12 14:22:27 ERROR 404: Not Found.

Done
Unpacking mecab-0.996-ko-0.9.2.tar.gz.......
Done
Change Directory to mecab-0.996-ko-0.9.2.......
install_mecab-ko_on_colab190912.sh: line 23: cd: mecab-0.996-ko-0.9.2/: No such file or directory
installing mecab-0.996-ko-0.9.2.tar.gz........
configure
make
make check
make install
ldconfig
Done
Change Directory to /content
from https:

In [41]:
# 실제 설치된 사전 경로 확인
!find /usr -name "dicrc" 2>/dev/null
!find /usr -name "mecab-ko*" -type d 2>/dev/null

/usr/share/mecab/dic/ipadic/dicrc
/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic/dicrc
/usr/local/lib/python3.12/dist-packages/mecab_ko_dic/dictionary/dicrc
/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic


In [42]:
from konlpy.tag import Mecab
mecab = Mecab('/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic')
print(mecab.morphs("대한민국의 수도는 서울입니다"))


['대한민국', '의', '수도', '는', '서울', '입니다']


In [43]:
%cd /content


/content


In [44]:
# 위키피디아 덤프(위키피디아 데이터) 다운로드
!wget https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2

--2026-03-12 14:22:53--  https://dumps.wikimedia.org/kowiki/latest/kowiki-latest-pages-articles.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1259021343 (1.2G) [application/octet-stream]
Saving to: ‘kowiki-latest-pages-articles.xml.bz2.3’

kowiki-latest-pages 100%[===================>]   1.17G  2.71MB/s    in 6m 24s  

2026-03-12 14:29:17 (3.12 MB/s) - ‘kowiki-latest-pages-articles.xml.bz2.3’ saved [1259021343/1259021343]



In [45]:
# 위키익스트랙터 를 이용한 위키피디아 덤프 파싱
!python -m wikiextractor.wikiextractor kowiki-latest-pages-articles.xml.bz2

/usr/bin/python3: No module named wikiextractor.wikiextractor


In [46]:
# 현재 경로에 있는 디렉터리와 파일들 의 리스트 받아오기
%ls

kowiki-latest-pages-articles.xml.bz2    Mecab-ko-for-Google-Colab/
kowiki-latest-pages-articles.xml.bz2.1  output_file.txt
kowiki-latest-pages-articles.xml.bz2.2  sample_data/
kowiki-latest-pages-articles.xml.bz2.3  text/


In [47]:
# 운영체제 기능 사용
import os
# 정규표현
import re

In [48]:
import os
print(os.listdir("/content"))

['.config', 'kowiki-latest-pages-articles.xml.bz2.3', 'text', 'output_file.txt', 'kowiki-latest-pages-articles.xml.bz2', 'Mecab-ko-for-Google-Colab', 'kowiki-latest-pages-articles.xml.bz2.1', 'kowiki-latest-pages-articles.xml.bz2.2', 'sample_data']


In [49]:
import os
import shutil

os.makedirs("/content/text", exist_ok=True)

shutil.copy(
    "/content/kowiki-latest-pages-articles.xml.bz2",
    "/content/text/kowiki-latest-pages-articles.xml.bz2"
)

print(os.listdir("/content/text"))

['kowiki-latest-pages-articles.xml.bz2']


In [64]:
os.listdir('text')



['kowiki-latest-pages-articles.xml.bz2']

In [51]:
# AA라는 디렉토리 파일 확인
# %ls text/AA

%ls
%ls text

kowiki-latest-pages-articles.xml.bz2    Mecab-ko-for-Google-Colab/
kowiki-latest-pages-articles.xml.bz2.1  output_file.txt
kowiki-latest-pages-articles.xml.bz2.2  sample_data/
kowiki-latest-pages-articles.xml.bz2.3  text/
kowiki-latest-pages-articles.xml.bz2


In [52]:
!pip install wikiextractor

In [53]:
# !python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text


In [54]:
import bz2

file_path = "/content/kowiki-latest-pages-articles.xml.bz2"

with bz2.open(file_path, "rt", encoding="utf-8", errors="ignore") as f:
    for i in range(20):
        print(f.readline())

<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.11/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.11/ http://www.mediawiki.org/xml/export-0.11.xsd" version="0.11" xml:lang="ko">

  <siteinfo>

    <sitename>위키백과</sitename>

    <dbname>kowiki</dbname>

    <base>https://ko.wikipedia.org/wiki/%EC%9C%84%ED%82%A4%EB%B0%B1%EA%B3%BC:%EB%8C%80%EB%AC%B8</base>

    <generator>MediaWiki 1.46.0-wmf.17</generator>

    <case>first-letter</case>

    <namespaces>

      <namespace key="-2" case="first-letter">미디어</namespace>

      <namespace key="-1" case="first-letter">특수</namespace>

      <namespace key="0" case="first-letter" />

      <namespace key="1" case="first-letter">토론</namespace>

      <namespace key="2" case="first-letter">사용자</namespace>

      <namespace key="3" case="first-letter">사용자토론</namespace>

      <namespace key="4" case="first-letter">위키백과</namespace>

      <namespace key="5" case="first-letter

반복되는 내용 확인

In [55]:
# AA- AF 디렉토리 안의 wiki 숫자 형태의 수많은 파일들을 하나로 통합하는 과정 진행
# AA~ AF 디렉토리 안 모든 파일들의 경로를 리스트 형태로 저장

import os
import re

def list_wiki(dirname):
    filepaths = []
    filenames = os.listdir(dirname)

    for filename in filenames:
        filepath = os.path.join(dirname, filename)

        if os.path.isdir(filepath):
            filepaths.extend(list_wiki(filepath))
        else:
            find = re.findall(r"wiki_[0-9][0-9]", filepath)
            if len(find) > 0:
                filepaths.append(filepath)

    return sorted(filepaths)

In [ ]:
!python -m wikiextractor.WikiExtractor /content/kowiki-latest-pages-articles.xml.bz2 -o /content/text


INFO: Preprocessing '/content/kowiki-latest-pages-articles.xml.bz2' to collect template definitions: this may take some time.


In [68]:
# 총 파일의 개수 확인
filepaths = list_wiki('text')
len(filepaths)

0

In [57]:
# output_file.txt 에 850개 파일을 전부 합치기
with open("output_file.txt", "w") as outfile:
  for filename in filepaths:
    with open(filename) as infile:
      contents = infile.read()
      outfile.write(contents)


In [58]:
f = open('output_file.txt', encoding='utf-8')

i = 0

while True:
  line = f.readline()
  if line != '\n':
    i = i+1
    print("%d번째 줄 : " %i + line)
  if i == 10:
    break
# open 을 했다면 꼭 닫아주어야 한다.
f.close()

1번째 줄 : 
2번째 줄 : 
3번째 줄 : 
4번째 줄 : 
5번째 줄 : 
6번째 줄 : 
7번째 줄 : 
8번째 줄 : 
9번째 줄 : 
10번째 줄 : 


In [61]:

# MeCab 바이너리 + 한국어 사전 설치
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev
!pip install mecab-python3
!pip install konlpy



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libmecab-dev is already the newest version (0.996-14build9).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-3).
mecab is already the newest version (0.996-14build9).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [60]:
import MeCab
mecab = MeCab.Tagger()
print(mecab.parse("테스트입니다"))

RuntimeError: 
----------------------------------------------------------

Failed initializing MeCab. Please see the README for possible solutions:

    https://github.com/SamuraiT/mecab-python3#common-issues

If you are still having trouble, please file an issue here, and include the
ERROR DETAILS below:

    https://github.com/SamuraiT/mecab-python3/issues

issueを英語で書く必要はありません。

------------------- ERROR DETAILS ------------------------
arguments: 
default dictionary path: None
[ifs] no such file or directory: /usr/local/lib/mecab/dic/mecab-ko-dic/dicrc
----------------------------------------------------------


In [ ]:
import MeCab
mecab = MeCab.Tagger()
print(mecab.parse("테스트입니다"))


In [ ]:
!apt-get install -y mecab libmecab-dev mecab-ipadic-utf8
!pip install mecab-python3

import subprocess
dic_path = subprocess.run(['mecab-config', '--dicdir'], capture_output=True, text=True).stdout.strip()
print("사전 경로:", dic_path)

In [ ]:
# 형태소 분석
from tqdm import tqdm
from konlpy.tag import Mecab

# Mecab을 사용한 토큰화 진행
mecab = Mecab()

# out_file 에는 총 몇줄이 있을까
f = open('output_file.txt', encoding='utf-8')
lines = f.read().splitlines()
print()

In [ ]:
# 상위 10 개만 출력
lines[:10]

In [ ]:
# 아무런 단어도 들어있지 않은 '' 와 같은 줄도 존재한다.
# 제외하고 형태소 분석을 수행한다.

result = []

for line in tqdm(lines):
  # 빈 문자열이 아닌 경우에만 수행
  if line:
    result.append(mecab.morphs(line))

In [ ]:
# 몇개의 문장이 존재 하고 , 얼마나 줄었는지
len(result)

In [ ]:
# 형태소 분석을 통해 토큰화 진행된 상태이므로 word2vec 을 학습
from gensim.models import word2ved
model = word2vec(result, size = 100, window = 5, min_count = 5, workers = 4, sg = 0)

In [ ]:
model_result1 = model.wv.most_similar("대한민국")
print(model_result1)

In [ ]:
model_result2 = model.wv.most_similar("어벤져스")
print(model_result2)

In [ ]:
model_result3 = model.wv.most_similar("반도체")
print(model_result3)